In [12]:
#Importar librerías y cargar los archivos
import pandas as pd
import numpy as np

# Rutas de los tres archivos
archivo_principal = "Datos_CLV_2026_Tec_v2.xlsx"
archivo_complementario = "Datos_Complementarios_CLV.xlsx"
archivo_entregas = "Relación de tiempos de entrega.xlsx"

# Cargamos las hojas del archivo principal
ventas = pd.read_excel(archivo_principal, sheet_name="Ventas_CLV_2022_2026")
existencias = pd.read_excel(archivo_principal, sheet_name="Existencias_04_09_2026")
compras = pd.read_excel(archivo_principal, sheet_name="Histórico_Compras")
salidas = pd.read_excel(archivo_principal, sheet_name="Salidas_Almacen")
clasificacion = pd.read_excel(archivo_principal, sheet_name="Clasificación_Articulos")

# Cargamos las hojas del archivo complementario
inv_complementario = pd.read_excel(
    archivo_complementario,
    sheet_name="Existencias_15_10_2026"
)

movimientos = pd.read_excel(
    archivo_complementario,
    sheet_name="Histórico_Entradas_SAlidas"
)

ventas_complementarias = pd.read_excel(
    archivo_complementario,
    sheet_name="Histórico_Ventas"
)

print("Archivos cargados correctamente")

Archivos cargados correctamente


In [13]:
#Normalizamos claves y columnas
# Función para normalizar identificadores
def limpiar_id(serie):
    return (
        serie.astype("string")
        .str.strip()
        .str.upper()
    )


# ==========================================
# 1. NORMALIZAR CLAVES DEL ARCHIVO PRINCIPAL
# ==========================================

for df, columnas in [
    (ventas, ["articulo_id", "clave_articulo"]),
    (existencias, ["articulo_id"]),
    (compras, ["articulo_id", "clave_articulo"]),
    (salidas, ["articulo_id", "clave_articulo"]),
    (clasificacion, ["articulo_id"])
]:

    for col in columnas:
        df[col] = limpiar_id(df[col])


# ==========================================
# 2. NORMALIZAR CLAVES COMPLEMENTARIAS
# ==========================================

# Inventario complementario
inv_complementario["articulo"] = limpiar_id(
    inv_complementario["articulo"]
)

# Movimientos históricos
movimientos["SKU"] = limpiar_id(
    movimientos["SKU"]
)

# Ventas complementarias
ventas_complementarias["articulo_id"] = limpiar_id(
    ventas_complementarias["articulo_id"]
)


# ==========================================
# 3. HOMOLOGAR NOMBRES DE COLUMNAS
# ==========================================

# Inventario: articulo -> articulo_id
inv_complementario = inv_complementario.rename(
    columns={
        "articulo": "articulo_id"
    }
)

# Movimientos: homologar nombre y marca
movimientos = movimientos.rename(
    columns={
        "NOMBRE": "nombre_articulo",
        "MARCA": "marca"
    }
)

# Ventas complementarias: homologar marca
ventas_complementarias = ventas_complementarias.rename(
    columns={
        "marc": "marca"
    }
)


# ==========================================
# 4. VERIFICAR RESULTADOS
# ==========================================

print("INVENTARIO COMPLEMENTARIO:")
print(inv_complementario.columns.tolist())

print("\nMOVIMIENTOS:")
print(movimientos.columns.tolist())

print("\nVENTAS COMPLEMENTARIAS:")
print(ventas_complementarias.columns.tolist())

print("\nCLASIFICACIÓN DE ARTÍCULOS:")
print(clasificacion.columns.tolist())

print("\nNormalización terminada correctamente.")



INVENTARIO COMPLEMENTARIO:
['articulo_id', 'nombre_articulo', 'marca', 'almacen', 'existencias', 'valor_inventario']

MOVIMIENTOS:
['ANO', 'MES', 'SKU', 'nombre_articulo', 'marca', 'SALIDAS', 'ALMACEN']

VENTAS COMPLEMENTARIAS:
['folio_abreviado', 'folio', 'fecha_documento', 'articulo_id', 'nombre_articulo', 'marca', 'precio_unitario', 'total_unidades', 'total_neto', 'nombre_cliente', 'estado    ', 'ciudad', 'cp', 'vendedor']

CLASIFICACIÓN DE ARTÍCULOS:
['articulo_id', 'nombre_articulo', 'categoria']

Normalización terminada correctamente.


In [14]:
#Construir un catálogo maestro de artículos
#Relacionamos articulo_id con clave_articulo. 
# Este catálogo será el puente para conectar los demás archivos.
# Construimos equivalencias desde las tablas principales


# 1. Construir equivalencias entre articulo_id y clave_articulo

equivalencias = pd.concat([
    ventas[["articulo_id", "clave_articulo"]],
    compras[["articulo_id", "clave_articulo"]],
    salidas[["articulo_id", "clave_articulo"]]
], ignore_index=True)

equivalencias = (
    equivalencias
    .dropna(subset=["articulo_id", "clave_articulo"])
    .drop_duplicates()
)

# 2. Identificar claves ambiguas

conflictos = (
    equivalencias.groupby("clave_articulo")["articulo_id"]
    .nunique()
)

claves_validas = conflictos[conflictos == 1].index

# 3. Construir catálogo de identificadores

catalogo = (
    equivalencias[
        equivalencias["clave_articulo"].isin(claves_validas)
    ]
    .drop_duplicates("clave_articulo")
)

# ==========================================
# 4. RECUPERAR NOMBRES DE LOS ARTÍCULOS
# ==========================================

# Utilizamos diferentes hojas como fuentes de nombres.
# Prioridad: clasificación, existencias, ventas,
# compras y salidas.

nombres_articulos = pd.concat([
    clasificacion[["articulo_id", "nombre_articulo"]],
    existencias[["articulo_id", "nombre_articulo"]],
    ventas[["articulo_id", "nombre_articulo"]],
    compras[["articulo_id", "nombre_articulo"]],
    salidas[["articulo_id", "nombre_articulo"]]
], ignore_index=True)

# Eliminar registros sin identificador o nombre

nombres_articulos = nombres_articulos.dropna(
    subset=["articulo_id", "nombre_articulo"]
)

# Conservar un nombre por articulo_id
# Se prioriza la primera fuente disponible

nombres_articulos = nombres_articulos.drop_duplicates(
    subset=["articulo_id"],
    keep="first"
)

# 5. Agregar nombre_articulo al catálogo

catalogo = catalogo.merge(
    nombres_articulos,
    on="articulo_id",
    how="left",
    validate="many_to_one"
)

# ==========================================
# 6. RESULTADOS
# ==========================================

print("Catálogo maestro construido correctamente")
print("Total de artículos:", len(catalogo))
print("Claves ambiguas:", (conflictos > 1).sum())
print(
    "Artículos sin nombre:",
    catalogo["nombre_articulo"].isna().sum()
)

display(catalogo.head(10))

Catálogo maestro construido correctamente
Total de artículos: 4979
Claves ambiguas: 0
Artículos sin nombre: 0


,articulo_id,clave_articulo,nombre_articulo
0,7688,MFTL374M2,MFT L37 4M2
1,7895,MFTPL313R2.5,MFT PL31 3R2.5
2,7985,MFTPU313R2.5,MFT PU31 3R2.5
3,8258,MFTT464M2,MFT T46 4M2
4,7613,MFTL332M2,MFT L33 2M2
5,7643,MFTL342M2,MFT L34 2M2
6,7673,MFTL372M2,MFT L37 2M2
7,7682,MFTL373M2,MFT L37 3M2
8,7706,MFTL37LN3L1.5,MFT L37LN 3L1.5
9,7709,MFTL37LN3L2.5,MFT L37LN 3L2.5


In [15]:
#Integrar los movimientos por SKU
#En lugar de unir los 55 mil movimientos directamente con las ventas,
# primero agrupamos los registros por SKU. 
# Así evitamos que una unión genere filas duplicadas.

# Convertir salidas a valores numéricos
movimientos["SALIDAS"] = pd.to_numeric(
    movimientos["SALIDAS"], errors="coerce"
).fillna(0)

# Agrupar movimientos por SKU
movimientos_agrupados = (
    movimientos.groupby("SKU", as_index=False)
    .agg(
        salidas_historicas=("SALIDAS", "sum"),
        registros_movimientos=("SKU", "size")
    )
)

# Relacionar SKU con clave_articulo
movimientos_relacionados = movimientos_agrupados.merge(
    catalogo,
    left_on="SKU",
    right_on="clave_articulo",
    how="left"
)

# Revisar cuántos SKU no encontraron correspondencia
print(
    "SKU sin correspondencia:",
    movimientos_relacionados["articulo_id"].isna().sum()
)

# Eliminar registros sin articulo_id
movimientos_relacionados = movimientos_relacionados.dropna(
    subset=["articulo_id"]
)

# Agrupar nuevamente por articulo_id
movimientos_por_articulo = (
    movimientos_relacionados.groupby("articulo_id", as_index=False)
    .agg(
        salidas_historicas=("salidas_historicas", "sum"),
        registros_movimientos=("registros_movimientos", "sum")
    )
)

print("Movimientos relacionados correctamente")
print("Artículos identificados:", len(movimientos_por_articulo))

display(movimientos_por_articulo.head())

SKU sin correspondencia: 569
Movimientos relacionados correctamente
Artículos identificados: 4724


,articulo_id,salidas_historicas,registros_movimientos
0,10004,4,4
1,10007,5,5
2,10010,17,8
3,10013,13,3
4,10016,20,11


In [16]:
#Preparar las tablas de ventas, compras e inventarios
#Agrupamos cada tabla a nivel de artículo antes de integrarla.

# VENTAS PRINCIPALES
ventas["total_unidades"] = pd.to_numeric(
    ventas["total_unidades"], errors="coerce"
).fillna(0)

ventas["total_neto"] = pd.to_numeric(
    ventas["total_neto"], errors="coerce"
).fillna(0)

ventas_agrupadas = (
    ventas.groupby("articulo_id", as_index=False)
    .agg(
        unidades_vendidas=("total_unidades", "sum"),
        importe_ventas=("total_neto", "sum"),
        numero_registros_venta=("articulo_id", "size")
    )
)

# COMPRAS PRINCIPALES
compras["unidades"] = pd.to_numeric(
    compras["unidades"], errors="coerce"
).fillna(0)

compras["precio_total_neto"] = pd.to_numeric(
    compras["precio_total_neto"], errors="coerce"
).fillna(0)

compras_agrupadas = (
    compras.groupby("articulo_id", as_index=False)
    .agg(
        unidades_compradas=("unidades", "sum"),
        importe_compras=("precio_total_neto", "sum"),
        numero_registros_compra=("articulo_id", "size")
    )
)

# SALIDAS DE ALMACÉN PRINCIPALES
salidas["salidas"] = pd.to_numeric(
    salidas["salidas"], errors="coerce"
).fillna(0)

salidas_agrupadas = (
    salidas.groupby("articulo_id", as_index=False)
    .agg(
        salidas_almacen=("salidas", "sum"),
        numero_registros_salidas=("articulo_id", "size")
    )
)

# EXISTENCIAS PRINCIPALES
existencias_agrupadas = existencias[
    ["articulo_id", "existencias", "valor_total_inventario"]
].copy()

# INVENTARIO COMPLEMENTARIO
inv_complementario = inv_complementario.rename(
    columns={"articulo": "articulo_id"}
)

inv_complementario["articulo_id"] = limpiar_id(
    inv_complementario["articulo_id"]
)

inv_complementario_agrupado = (
    inv_complementario.groupby("articulo_id", as_index=False)
    .agg(
        existencias_15_oct=("existencias", "sum"),
        valor_inventario_15_oct=("valor_inventario", "sum")
    )
)

In [17]:
#Integramos las ventas complementarias

ventas_complementarias["total_unidades"] = pd.to_numeric(
    ventas_complementarias["total_unidades"], errors="coerce"
).fillna(0)

ventas_complementarias["total_neto"] = pd.to_numeric(
    ventas_complementarias["total_neto"], errors="coerce"
).fillna(0)

ventas_comp_agrupadas = (
    ventas_complementarias.groupby("articulo_id", as_index=False)
    .agg(
        unidades_ventas_complementarias=("total_unidades", "sum"),
        importe_ventas_complementarias=("total_neto", "sum"),
        registros_ventas_complementarias=("articulo_id", "size")
    )
)

In [18]:
#validamos los duplicados
tablas = {
    "Ventas": ventas_agrupadas,
    "Compras": compras_agrupadas,
    "Salidas": salidas_agrupadas,
    "Existencias": existencias_agrupadas,
    "Inventario complementario": inv_complementario_agrupado,
    "Ventas complementarias": ventas_comp_agrupadas,
    "Movimientos": movimientos_por_articulo
}

for nombre, df in tablas.items():

    nulos = df["articulo_id"].isna().sum()
    duplicados = df["articulo_id"].duplicated().sum()

    print(f"\nTabla: {nombre}")
    print("Registros:", len(df))
    print("Claves nulas:", nulos)
    print("Claves duplicadas:", duplicados)


Tabla: Ventas
Registros: 3895
Claves nulas: 0
Claves duplicadas: 0

Tabla: Compras
Registros: 3698
Claves nulas: 0
Claves duplicadas: 0

Tabla: Salidas
Registros: 4343
Claves nulas: 0
Claves duplicadas: 0

Tabla: Existencias
Registros: 2950
Claves nulas: 0
Claves duplicadas: 0

Tabla: Inventario complementario
Registros: 2951
Claves nulas: 0
Claves duplicadas: 0

Tabla: Ventas complementarias
Registros: 2130
Claves nulas: 0
Claves duplicadas: 0

Tabla: Movimientos
Registros: 4724
Claves nulas: 0
Claves duplicadas: 0


In [19]:
#Creamos la base de datos integrada
#Utilizaremos articulo_id como identificador central. 
# El catálogo maestro contiene las equivalencias con clave_articulo,
# que a su vez permite relacionar los SKU.

# Partimos del catálogo de artículos
base_integrada = catalogo.copy()

# Integrar las ventas principales
base_integrada = base_integrada.merge(
    ventas_agrupadas,
    on="articulo_id",
    how="left",
    validate="one_to_one"
)

# Integrar compras
base_integrada = base_integrada.merge(
    compras_agrupadas,
    on="articulo_id",
    how="left",
    validate="one_to_one"
)

# Integrar salidas de almacén
base_integrada = base_integrada.merge(
    salidas_agrupadas,
    on="articulo_id",
    how="left",
    validate="one_to_one"
)

# Integrar existencias principales
base_integrada = base_integrada.merge(
    existencias_agrupadas,
    on="articulo_id",
    how="left",
    validate="one_to_one"
)

# Integrar inventario complementario
base_integrada = base_integrada.merge(
    inv_complementario_agrupado,
    on="articulo_id",
    how="left",
    validate="one_to_one"
)

# Integrar ventas complementarias
base_integrada = base_integrada.merge(
    ventas_comp_agrupadas,
    on="articulo_id",
    how="left",
    validate="one_to_one"
)

# Integrar movimientos históricos
base_integrada = base_integrada.merge(
    movimientos_por_articulo,
    on="articulo_id",
    how="left",
    validate="one_to_one"
)

# Mostrar resultado
print("Dimensiones de la base integrada:", base_integrada.shape)

base_integrada.head()

Dimensiones de la base integrada: (4979, 20)


,articulo_id,clave_articulo,nombre_articulo,unidades_vendidas,importe_ventas,numero_registros_venta,unidades_compradas,importe_compras,numero_registros_compra,salidas_almacen,numero_registros_salidas,existencias,valor_total_inventario,existencias_15_oct,valor_inventario_15_oct,unidades_ventas_complementarias,importe_ventas_complementarias,registros_ventas_complementarias,salidas_historicas,registros_movimientos
0,7688,MFTL374M2,MFT L37 4M2,133.0,19413.38,69,170.0,12889.18,3,117.0,24,NaN,NaN,NaN,NaN,24.0,3455.05,11,251.0,43
1,7895,MFTPL313R2.5,MFT PL31 3R2.5,735.0,106745.62,271,1084.0,87476.87,9,661.0,30,19.0,1496.30,7.0,551.27,115.0,16211.23,38,1286.0,60
2,7985,MFTPU313R2.5,MFT PU31 3R2.5,860.0,125002.04,317,1326.0,107052.63,9,756.0,30,15.0,1181.29,NaN,NaN,168.0,24297.38,60,1467.0,62
3,8258,MFTT464M2,MFT T46 4M2,146.0,21765.09,84,285.0,22124.83,3,120.0,30,60.0,4652.70,57.0,4420.06,41.0,6014.73,18,274.0,57
4,7613,MFTL332M2,MFT L33 2M2,966.0,140325.25,364,1170.0,93784.75,6,834.0,30,40.0,3246.32,NaN,NaN,249.0,36526.37,82,1934.0,63


In [20]:
#Validamos y exportamos
# Revisar duplicados de la clave principal
print(
    "Artículos duplicados:",
    base_integrada["articulo_id"].duplicated().sum()
)

# Revisar valores faltantes
print("\nValores faltantes:")
print(base_integrada.isnull().sum())

# Exportar
base_integrada.to_excel(
    "Base_Consolidada_Articulos_Versión_Complementaria.xlsx",
    index=False
)

print("Base integrada exportada correctamente")

Artículos duplicados: 0

Valores faltantes:
articulo_id                            0
clave_articulo                         0
nombre_articulo                        0
unidades_vendidas                   1084
importe_ventas                      1084
numero_registros_venta              1084
unidades_compradas                  1281
importe_compras                     1281
numero_registros_compra             1281
salidas_almacen                      636
numero_registros_salidas             636
existencias                         2421
valor_total_inventario              2421
existencias_15_oct                  2478
valor_inventario_15_oct             2478
unidades_ventas_complementarias     2863
importe_ventas_complementarias      2863
registros_ventas_complementarias    2863
salidas_historicas                   255
registros_movimientos                255
dtype: int64
Base integrada exportada correctamente
